# பாடம் 05 - முகாமைப் பண்புடைய RAG


## அமைப்பு

இந்த நோட்புக் மைக்ரோசாஃப்ட் ஏஜென்ட் 框架 மூலம் Agentic RAG (Retrieval-Augmented Generation) மாதிரியை காட்சி alaṉṟu செய்கிறது.

**முன்னுட்பங்கள்:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — உங்களது Azure AI Search சேவை இறுதி புள்ளி
- `AZURE_SEARCH_API_KEY` — உங்களது Azure AI Search API விசை
- சூழல் மாறிலிகள் மூலம் அமைக்கப்பட்ட Azure OpenAI நிறைவு
- Azure CLI அங்கீகரிக்கப்பட்டது (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ஏஜென்டிக் RAG என்றால் என்ன?

சම්பிரதாயீக RAG ஒரு நிர்ணயிக்கப்பட்ட செயல்முறையை பின்பற்றுகிறது: ஆவணங்களை பெறவும், பின்னர் ஒரு பதிலை உருவாக்கவும். **ஏஜென்டிக் RAG** இதனை மேலும் விரிவுபடுத்தி, தகவலை எப்போது மற்றும் எவ்வாறு பெறுவது என்பது குறித்து ஏஜெண்டுக்கு சுயாதீனத்தன்மையை அளிக்கிறது.

ஏஜென்டிக் RAG உடன், ஏஜெண்ட் செய்யக்கூடியவை:
- ஒரு கேள்விக்கு பதிலளிப்பதற்கு முன்பு தகவல் பெறுவது தேவையா என்பதை **தீர்மானிக்க**  
- எந்த தரவுத் தளம் அல்லது கருவியை கேள்வி செய்வது என்பது குறித்து **தேர்வு செய்ய**  
- பெறப்பட்ட முடிவுகளை **மதிப்பாய்வு செய்து**, முதல் முயற்சி போதுமானது அல்ல எனில் தொடர்ந்த தகவல் பெறுதலைச் செய்ய  
- பல தகவல் பெறுதல் படிகளிலிருந்து உள்ளடக்கத்தைச் சேர்த்து ஒருங்கிணைந்த பதிலை உருவாக்க  

இது ஓர் நிலையான பெறுதல்-பின்னர்-உருவாக்க முன்மாதிரையைவிட, ஏஜெண்டை நன்றாக uyeழைக்கும் மற்றும் துல்லியமானதாக மாற்றுகிறது.


## தேடல் கருவியை உருவாக்குதல்

Agentic RAG இல், வெளிப்புற தரவு ஆதாரங்கள் முகாம் கோரியவாறு முகாமையாளர் அழைக்கக்கூடிய **கருவிகள்** என்று பூட்டப்படுகின்றன. இது முகாமையாளருக்கு மருவிப்பை அவசியமான படி அல்லாமல், மேற்கொள்ளக்கூடிய மற்றொரு செயல் என்று கருத அனுமதிக்கின்றது.

இதற்கு கீழே நாம் பயண அறிவுக் களஞ்சியத்தை வரையறுக்கிறோம் மற்றும் அது முகாமையாளர் நோக்கி இடத்தைப் பார்க்க அழைக்கக்கூடிய ஒரு கருவியாக வெளிப்படுத்துகிறோம்.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG முகவரியை உருவாக்குதல்

இப்போது நாம் எதற்கு பதில் அளிக்குமுன் **எப்போதும் தகவலை மீட்டெடுக்கக் கட்டளையிடப்பட்டுள்ள முகவரியை** உருவாக்குகிறோம். இந்த முகவர் தனது பதில்களை தனது பயிற்சி தரவுகளுக்கு பதிலாக அறிவு தரவுத்தளத்தில் நிலைநிறுத்த `search_travel_knowledge` கருவியை பயன்படுத்துகிறது.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## முறையாக மீட்டம் — உருவாக்குனர்-தயாரிப்பாளர் முறை

Agentic RAG இன் ஒரு முக்கிய நன்மை **முறையான மீட்டம்** ஆகும். முகவர் தன் ஆரம்ப கண்டுபிடிப்புகளை உறுதிப்படுத்த, சீரமைக்க அல்லது விரிவுபடுத்த பல சுற்றுகளாக தேடலை செய்ய முடியும் — இது "உருவாக்குனர்-தயாரிப்பாளர்" வேலைபாட்டைப் போலவே:

1. **உருவாக்குனர் படி**: முகவர் ஆரம்ப தகவல்களை மீட்டெடுக்கிறான் மற்றும் ஒரு பதிலை உருவாக்குகிறான்.
2. **தயாரிப்பாளர் படி**: முகவர் தகவல்களை உறுதிப்படுத்த அல்லது குறைகள் நிரப்ப கூடுதல் மீட்டங்களை செய்யும்.

கீழே, முகவரிடம் பல இடங்களைக் ஒப்பிடும் கேள்வி கேட்கப்பட்டது, அதனால் அது பலமுறை தேடல் செய்ய வேண்டியவாறு இருக்கிறது.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## 요약

이 수업에서는 Microsoft 에이전트 프레임워크를 사용하여 **Agentic RAG** 시스템을 구축하는 방법을 배웠습니다:

- **Agentic RAG** 는 에이전트가 언제 정보를 검색할지 자율적으로 결정하도록 하여, 검색을 고정된 것이 아니라 동적으로 만듭니다.
- **도구를 데이터 소스로 사용**: 외부 지식 기반(예: Azure AI Search)이 에이전트가 호출할 수 있는 도구로 감싸집니다.
- **반복 검색**: 메이커-체커 패턴을 통해 에이전트가 최종 답변을 생성하기 전에 여러 번 검색, 검증, 정제 단계를 수행할 수 있습니다.

실제 환경에서는 메모리 내 `TRAVEL_KNOWLEDGE_BASE`를 대규모 여행 문서 검색을 처리할 수 있는 실제 Azure AI Search 인덱스로 교체합니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**கவனிக்கை**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவையாகும் [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழி ப译ப்பட்டது. நாங்கள் துல்லியத்தை உறுதிபடுத்த முயற்சித்தாலும், தானியங்கி மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கக்கூடும் என்பதனை தயவுசெய்து கவனத்தில் கொள்ளவும். இதுৎமை மொழியில் உள்ள அசல் ஆவணம் அதிகாரப்பூர்வ இருப்பாகக் கருதப்பட வேண்டும். முக்கிய தகவல்களுக்கு தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்தியதರಿಂದ உண்டாகக்கூடிய எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்கு நாங்கள் பொறுப்பேற்க மாட்டோம்.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
